# Project 1: Swim with Invariant Filters

In this project, you will implement an Extended Kalman Filter (EKF) to localize a simulated underwater robot using IMU (prediction) and Range Sensor measurements (correction). Additionally, implement simple trajectory tracking control. This should be both educational and fun!

This project requires use of Webots on macOS or Linux. For Windows users, you'll need to set up WSL2 (refer to [the guide on Piazza](https://piazza.com/class/mk62vbxt9lq5dp/post/14)).

In [1]:
import gtsam
import numpy as np
from tests.utils import verify, setup_path
setup_path()

# Part 1: Action (Keyboard Control)
In this part, you will implement keyboard-based control for the robot by mapping user inputs to thruster commands. You will work exclusively in `brain.py`, specifically within the `act_on_command` and `wrench_to_thrusters` functions.

### Tasks

1. **Implement `act_on_command`** in `brain.py`
   - Map keyboard inputs to the appropriate wrench using the provided step-size constants (`FORCE_X_STEP`, `FORCE_Z_STEP`, `TAU_Z_STEP`, `TAU_X_STEP`)
   - The key mappings are:
     - **UP/DOWN arrows**: Forward/backward force (±X)
     - **W/S keys**: Up/down force (±Z)
     - **LEFT/RIGHT arrows**: Yaw torque (turning)
     - **Q/E keys**: Roll torque

2. **Implement `wrench_to_thrusters`** in `brain.py`
   - This function converts a desired wrench (forces and torques) into four thruster commands
   - Use the provided `INVERSE_B_MATRIX` to perform the conversion
   - The thruster mapping is: `(u_lm, u_rm, u_vlm, u_vrm)` for left motor, right motor, vertical left motor, and vertical right motor

### Testing

- **Run the cell** below. These tests will also be run by the auto-grader (in addition to additional tests). 

### Verification in Webots

Once Part 1 is complete, you should be able to control the robot manually. You can verify the behavior in the Webots simulation:
- Ensure the simulation is in **keyboard mode**.
- Click on the simulation window to focus it before using the keys.
- **Answer the reflection questions** in the PowerPoint (project_1c_reflection.pptx). You will need to paste a screenshot there as well.



In [2]:
from tests.part1_action import test_wrench_to_thrusters, test_act_on_command

print("test_wrench_to_thrusters:", verify(test_wrench_to_thrusters))
print("test_act_on_command:", verify(test_act_on_command))

test_wrench_to_thrusters: "Correct"
test_act_on_command: "Correct"


# Part 2: Perception
In this part, you're building an EKF-based state estimator for the underwater robot.

You'll be working in `brain.py` to implement the filtering logic. The class that you need to use in GT-SAM is the specialized extended Kalman filter on NavState that we discussed in class: the [NavStateImuEKF](https://borglab.github.io/gtsam/navstateimuekf/).

### Tasks

1. **Implement `initialize_EKF`** in `brain.py`
   - Create the EKF filter using the provided initial state `X0`, covariance `P0`, and `PREINTEGRATION_PARAMS`
   - Initialize `self.ekf` as a `NavStateImuEKF` object

2. **Implement `EKF_predict`** in `brain.py`
   - Handle the prediction step using gyroscope (`omega_meas`) and accelerometer (`acc_meas`) readings
   - Integrate the motion forward using these inputs and update the state estimate and its covariance 
   - Use `self.ekf.predict()` with the provided measurements and time step `dt`
   - Return the updated state and covariance
   - The exact method you will use is a specialized version of predict that takes the IMU readings and its arguments are described in detail [here](https://github.com/borglab/gtsam/blob/e155fe616f802561d40ab8b0319fcc5ea0b2d7c2/gtsam/navigation/NavStateImuEKF.h#L78C1-L90C1)

3. **Implement `EKF_update`** in `brain.py`
   - Apply measurement updates to correct your estimate based on available sensor data
   - Handle three types of simulated measurements that might be available (these measurements arrive at different cadences, so check which ones are available before applying any update.):
     - **Position measurements**: Direct position observations
     - **Depth measurements**: Single depth (z-coordinate) observation
     - **Range measurements**: Distances to three fixed beacon locations
   - For each measurement type, compute the appropriate measurement matrix `H` and call `self.ekf.updateWithVector`
   - To complete this part, you will need to understand the method `updateWithVector`. Luckily, GTSAM [provides a user guide on exactly this](https://borglab.github.io/gtsam/ekf-variants/#navstateimuekf-navstate-imu-ekf). The arguments are described in detail [here](https://github.com/borglab/gtsam/blob/e155fe616f802561d40ab8b0319fcc5ea0b2d7c2/gtsam/navigation/ManifoldEKF.h#L218C1-L229C1).

### Testing

- **Run the cell** below. These tests will also be run by the auto-grader (in addition to additional tests). 

### Verification in Webots

Once Part 2 is complete, you should be able to plot estimations and visualize the results using the following keys:
- Press **A** to plot attitude (roll/pitch/yaw)
- Press **P** to plot position
- Press **V** to plot velocity


In [3]:
from tests.part2_perception import (
    test_initialize_EKF,
    test_EKF_predict,
    test_EKF_predict_update,
)

print("test_initialize_EKF:", verify(test_initialize_EKF))
print("test_EKF_predict:", verify(test_EKF_predict))
print("test_EKF_predict_update:", verify(test_EKF_predict_update))

test_initialize_EKF: "Correct"
test_EKF_predict: "Correct"
test_EKF_predict_update: "Wrong"


In [4]:
# Debug EKF_predict_update
import numpy as np
from controllers.ROV_controller.brain import Brain
from controllers.ROV_controller.robot import BEACONS, Measurements

inputs = np.load("tests/test_data/ekf_inputs.npz", allow_pickle=True)
outputs = np.load("tests/test_data/ekf_outputs.npz", allow_pickle=True)

# Test first case
c_in = inputs["arr_0"].item()
c_out = outputs["arr_0"].item()

brain = Brain(BEACONS)

R = gtsam.Rot3(c_in["X0_pose"][:3, :3])
t = gtsam.Point3(c_in["X0_pose"][:3, 3])
X0 = gtsam.NavState(gtsam.Pose3(R, t), gtsam.Point3(c_in["X0_vel"]))

brain.initialize_EKF(X0, c_in["P0"])

# Predict
X_pred, P_pred = brain.EKF_predict(c_in["omega"], c_in["acc"], c_in["dt"])

print("After predict:")
print("Position error:", np.linalg.norm(X_pred.position() - c_out["X_pred"]["p"]))
print("Velocity error:", np.linalg.norm(X_pred.velocity() - c_out["X_pred"]["v"]))
print()

# Update
meas = Measurements(
    position=c_in["meas"]["position"],
    depth=c_in["meas"]["depth"],
    ranges=c_in["meas"]["ranges"],
    X_true=None,
)

print("Measurements available:")
print("  position:", meas.position is not None, meas.position if meas.position is not None else "")
print("  depth:", meas.depth is not None, meas.depth if meas.depth is not None else "")
print("  ranges:", meas.ranges is not None, meas.ranges if meas.ranges is not None else "")
print()

brain.EKF_update(meas)

X_upd = brain.ekf.state()
P_upd = brain.ekf.covariance()

print("After update:")
print("Expected position:", c_out["X_upd"]["p"])
print("Got position:     ", X_upd.position())
print("Position error:", np.linalg.norm(X_upd.position() - c_out["X_upd"]["p"]))
print()
print("Expected velocity:", c_out["X_upd"]["v"])
print("Got velocity:     ", X_upd.velocity())
print("Velocity error:", np.linalg.norm(X_upd.velocity() - c_out["X_upd"]["v"]))
print()
print("Covariance error:", np.linalg.norm(P_upd - c_out["P_upd"]))

After predict:
Position error: 0.0
Velocity error: 0.0

Measurements available:
  position: True [ 0.20053171 -0.09901583  0.04999998]
  depth: False 
  ranges: False 

After update:
Expected position: [ 0.01893476 -0.0082176   0.00459082]
Got position:      [ 0.01897144 -0.00816691  0.00454489]
Position error: 7.761501531252644e-05

Expected velocity: [0.01363933 0.01817593 0.00045235]
Got velocity:      [0.01364471 0.01818339 0.00044774]
Velocity error: 1.0285604325794287e-05

Covariance error: 0.0051990019799323705


In [5]:
# Check ALL test cases
inputs = np.load("tests/test_data/ekf_inputs.npz", allow_pickle=True)
outputs = np.load("tests/test_data/ekf_outputs.npz", allow_pickle=True)

print(f"Total test cases: {len(inputs.files)}")
print()

for i in range(len(inputs.files)):
    c_in = inputs[f"arr_{i}"].item()
    c_out = outputs[f"arr_{i}"].item()
    
    brain = Brain(BEACONS)
    
    R = gtsam.Rot3(c_in["X0_pose"][:3, :3])
    t = gtsam.Point3(c_in["X0_pose"][:3, 3])
    X0 = gtsam.NavState(gtsam.Pose3(R, t), gtsam.Point3(c_in["X0_vel"]))
    
    brain.initialize_EKF(X0, c_in["P0"])
    X_pred, P_pred = brain.EKF_predict(c_in["omega"], c_in["acc"], c_in["dt"])
    
    meas = Measurements(
        position=c_in["meas"]["position"],
        depth=c_in["meas"]["depth"],
        ranges=c_in["meas"]["ranges"],
        X_true=None,
    )
    
    brain.EKF_update(meas)
    
    X_upd = brain.ekf.state()
    P_upd = brain.ekf.covariance()
    
    # Check tolerances
    pos_err = np.linalg.norm(X_upd.position() - c_out["X_upd"]["p"])
    vel_err = np.linalg.norm(X_upd.velocity() - c_out["X_upd"]["v"])
    cov_err = np.linalg.norm(P_upd - c_out["P_upd"])
    
    pos_pass = pos_err < 1e-3
    vel_pass = vel_err < 1e-3
    cov_pass = cov_err < 5e-2
    
    all_pass = pos_pass and vel_pass and cov_pass
    
    print(f"Test {i}: {'✓ PASS' if all_pass else '✗ FAIL'}")
    print(f"  Measurements: pos={c_in['meas']['position'] is not None}, depth={c_in['meas']['depth'] is not None}, ranges={c_in['meas']['ranges'] is not None}")
    if not all_pass:
        print(f"  Position error: {pos_err:.6f} {'✓' if pos_pass else '✗ FAIL'}")
        print(f"  Velocity error: {vel_err:.6f} {'✓' if vel_pass else '✗ FAIL'}")
        print(f"  Covariance error: {cov_err:.6f} {'✓' if cov_pass else '✗ FAIL'}")
    print()

Total test cases: 6

Test 0: ✓ PASS
  Measurements: pos=True, depth=False, ranges=False

Test 1: ✓ PASS
  Measurements: pos=False, depth=True, ranges=False

Test 2: ✓ PASS
  Measurements: pos=False, depth=False, ranges=True

Test 3: ✗ FAIL
  Measurements: pos=True, depth=False, ranges=False
  Position error: 0.019517 ✗ FAIL
  Velocity error: 0.010576 ✗ FAIL
  Covariance error: 0.027958 ✓

Test 4: ✗ FAIL
  Measurements: pos=False, depth=True, ranges=False
  Position error: 0.000196 ✓
  Velocity error: 0.000106 ✓
  Covariance error: 0.062573 ✗ FAIL

Test 5: ✗ FAIL
  Measurements: pos=False, depth=False, ranges=True
  Position error: 0.003068 ✗ FAIL
  Velocity error: 0.001399 ✗ FAIL
  Covariance error: 0.033075 ✓



In [6]:
# Deep dive into failing test case 3
i = 3
c_in = inputs[f"arr_{i}"].item()
c_out = outputs[f"arr_{i}"].item()

brain = Brain(BEACONS)

R = gtsam.Rot3(c_in["X0_pose"][:3, :3])
t = gtsam.Point3(c_in["X0_pose"][:3, 3])
X0 = gtsam.NavState(gtsam.Pose3(R, t), gtsam.Point3(c_in["X0_vel"]))

brain.initialize_EKF(X0, c_in["P0"])
X_pred, P_pred = brain.EKF_predict(c_in["omega"], c_in["acc"], c_in["dt"])

print(f"Test {i} - Deep Dive")
print("="*50)
print("Predicted state (after predict, before update):")
print(f"  Position: {X_pred.position()}")
print(f"  Expected after update: {c_out['X_upd']['p']}")
print()
print(f"Position measurement: {c_in['meas']['position']}")
print(f"Measurement difference from prediction: {c_in['meas']['position'] - X_pred.position()}")
print()

meas = Measurements(
    position=c_in["meas"]["position"],
    depth=c_in["meas"]["depth"],
    ranges=c_in["meas"]["ranges"],
    X_true=None,
)

# Manual check of what updateWithVector should receive
X_est = brain.ekf.state()
H = np.zeros((3, 9))
H[0:3, 3:6] = np.eye(3)
z = meas.position
h = X_est.position()

print(f"Update parameters:")
print(f"  h (predicted measurement): {h}")
print(f"  z (actual measurement): {z}")
print(f"  innovation (z - h): {z - h}")
print()

brain.EKF_update(meas)
X_upd = brain.ekf.state()

print(f"After update:")
print(f"  Got position: {X_upd.position()}")
print(f"  Expected:     {c_out['X_upd']['p']}")
print(f"  Error: {np.linalg.norm(X_upd.position() - c_out['X_upd']['p'])}")

Test 3 - Deep Dive
Predicted state (after predict, before update):
  Position: [1.10005226 1.90017744 2.99999987]
  Expected after update: [1.13556482 1.88242089 3.00861619]

Position measurement: [1.30005226 1.80017744 3.04999987]
Measurement difference from prediction: [ 0.2  -0.1   0.05]

Update parameters:
  h (predicted measurement): [1.10005226 1.90017744 2.99999987]
  z (actual measurement): [1.30005226 1.80017744 3.04999987]
  innovation (z - h): [ 0.2  -0.1   0.05]

After update:
  Got position: [1.13974285 1.90148555 3.00858135]
  Expected:     [1.13556482 1.88242089 3.00861619]
  Error: 0.019517129022607792


# Part 3: Control (Trajectory Following)

In this part, you will implement a controller to follow a desired trajectory. You won't be implementing a full PID controller: we were able to control the vehicle just fine with just a proportional control signal. You will work in `brain.py`, where you need to implement the function `follow_step`.

In Part 1, we implemented keyboard control in 4 different directions. Each of these 4 different directions corresponded to a particular wrench (2 pure torque and 2 pure force wrenches). Here we take the same view, but we need to wrap a simple controller around each of these same four directions. Think about how to compute the error for each of these four basic movements. Note that these errors need to be expressed in the body coordinate frame.

Some guidance about angles: the desired yaw or heading needs to be derived from the desired state, but the desired roll is always zero. One thing to watch out for is that differences in angle have to be wrapped to a standard interval for which we provide a function `wrap_to_pi`. 

### Tasks

1. **Implement `follow_step`** in `brain.py`
   - Compute control commands to track a desired trajectory given the current state `X` and `desired_state`
   - Use the provided proportional gains: `KP_DIS`, `KP_YAW`, `KP_Z`, `KP_ROLL`
   - Compute errors in:
     - XY-plane position (distance error)
     - Z-axis position (depth error)
     - Yaw angle (the desired yaw/heading needs to be derived from the desired state)
     - Roll angle (the desired roll is always zero)
   - Remember to wrap angular errors using `self.wrap_to_pi()`. Differences in angle have to be wrapped to a standard interval
   - Convert the computed forces/torques to thruster commands using `wrench_to_thrusters`
   - Feel free to implement whatever controller you want. We have given some *proportional* gain constants (e.g., `KP_DIS` for distance error in the forward direction) that worked well with how we defined the error. Your mileage may vary.


### Testing

- **Run the cell** below. These tests will also be run by the auto-grader (in addition to additional tests). 

### Verification in Webots

Once Part 3 is complete, you should see the robot following the desired trajectory in Webots using **trajectory following mode**.


In [25]:
from tests.part3_control import test_follow_step
print("test_follow_step:", verify(test_follow_step))
test_follow_step()

test_follow_step: "Wrong"


AssertionError: 

# Tips and Common Issues

- **Test frequently**: Run the notebook tests after implementing each function to catch errors early
- **Check your matrix operations**: Make sure you understand the relationship between wrench and thruster commands (the `B_MATRIX` mapping)
- **Angular wrapping**: Always use `wrap_to_pi()` when computing angular errors to handle the discontinuity at ±π
- **EKF measurement updates**: Remember that measurements arrive at different rates. Always check if a measurement is available (`is not None`) before using it
- **Webots refresh**: After changing Python code, press the refresh button in Webots to reload your controller 

# Next Step: Reflection Questions

Complete the reflection questions in the provided PowerPoint template (`project_1c_reflection.pptx`). Provide your answers on the appropriate slides. Do not delete or add new slides.